# Predicting household electricity consumption (kW)

In [2]:
# Import Libraries

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, accuracy_score, roc_curve, roc_auc_score


In [56]:
# Getting the data
df = pd.read_csv('data/household_power_consumption.txt',
                 sep=";",
                 low_memory=False)

### Data Cleaning

In [57]:
print(df.shape)

(2075259, 9)


In [58]:
# Checking the null values 
print(df.isnull().sum())

Date                         0
Time                         0
Global_active_power          0
Global_reactive_power        0
Voltage                      0
Global_intensity             0
Sub_metering_1               0
Sub_metering_2               0
Sub_metering_3           25979
dtype: int64


In [59]:
# Checking for missing values
for col in df.columns:
    n = (df[col].astype(str) == "?").sum()
    if n > 0:
        print(f"{col}: {n} literal '?' values")

Global_active_power: 25979 literal '?' values
Global_reactive_power: 25979 literal '?' values
Voltage: 25979 literal '?' values
Global_intensity: 25979 literal '?' values
Sub_metering_1: 25979 literal '?' values
Sub_metering_2: 25979 literal '?' values


### Important Note:
We can notice that for the Sub_metering_3 column we don't have any "?" and after checking the raw data for the rows that have "?" across all columns, we noticed that we have 8 values and not 9 (e.g 13/1/2010;12:50:00;?;?;?;?;?;?;).\
So we decided to reload the data line by line and fix this issue by replacing the "?" with NaN and adding NaN also for the missing values of Sub_metering_3.

In [60]:
cols = ["Date","Time","Global_active_power","Global_reactive_power","Voltage",
        "Global_intensity","Sub_metering_1","Sub_metering_2","Sub_metering_3"]

rows = []
with open("data/household_power_consumption.txt", "r") as f:
    header = f.readline()                      # skip the header line
    for line in f:
        parts = line.rstrip("\n").split(";")
        # pad to 9 fields: a trailing ';' means the last field is absent
        while len(parts) < len(cols):
            parts.append(np.nan)
        # any literal '?' becomes a real missing value
        parts = [np.nan if p in ("?", "") else p for p in parts]
        rows.append(parts)

df = pd.DataFrame(rows, columns=cols)

In [61]:
# verification
print(df.shape)
print(df.dtypes)
print(df.isnull().sum())

(2075259, 9)
Date                     object
Time                     object
Global_active_power      object
Global_reactive_power    object
Voltage                  object
Global_intensity         object
Sub_metering_1           object
Sub_metering_2           object
Sub_metering_3           object
dtype: object
Date                         0
Time                         0
Global_active_power      25979
Global_reactive_power    25979
Voltage                  25979
Global_intensity         25979
Sub_metering_1           25979
Sub_metering_2           25979
Sub_metering_3           25979
dtype: int64


In [62]:
# Convert the columns to numeric
num_cols = [
    "Global_active_power","Global_reactive_power","Voltage",
    "Global_intensity","Sub_metering_1","Sub_metering_2","Sub_metering_3"
]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

In [63]:
# check if the 25979 nulls are all in the same rows
print("Rows where all 7 '?' columns are null:",
      df[["Global_active_power","Global_reactive_power","Voltage",
        "Global_intensity","Sub_metering_1","Sub_metering_2","Sub_metering_3"]].isnull().all(axis=1).sum())

Rows where all 7 '?' columns are null: 25979


In [64]:
# Drop those null rows
df = df.dropna(subset=num_cols)

In [ ]:
# Combine the 'date' and 'time' in one column
df["datetime"] = pd.to_datetime(df["Date"] + " " + df["Time"],
                                format="%d/%m/%Y %H:%M:%S")
df = df.drop(columns=["Date","Time"])

In [72]:
# Verification
print(df.shape)
print(df.dtypes)

(2049280, 8)
datetime                 datetime64[ns]
Global_active_power             float64
Global_reactive_power           float64
Voltage                         float64
Global_intensity                float64
Sub_metering_1                  float64
Sub_metering_2                  float64
Sub_metering_3                  float64
dtype: object


In [73]:
# Check for duplicate timestamps and full duplicate rows
print("Duplicate datetimes:", df["datetime"].duplicated().sum())
print("Duplicate full rows:", df.duplicated().sum())

Duplicate datetimes: 0
Duplicate full rows: 0


In [74]:
num_cols = ["Global_active_power","Global_reactive_power","Voltage",
            "Global_intensity","Sub_metering_1","Sub_metering_2","Sub_metering_3"]

for c in num_cols:
    n = (df[c] == 0).sum()
    print(f"{c}: {n} zeros ({n/len(df)*100:.2f}%)")

Global_active_power: 0 zeros (0.00%)
Global_reactive_power: 481561 zeros (23.50%)
Voltage: 0 zeros (0.00%)
Global_intensity: 0 zeros (0.00%)
Sub_metering_1: 1880175 zeros (91.75%)
Sub_metering_2: 1436830 zeros (70.11%)
Sub_metering_3: 852092 zeros (41.58%)


In [ ]:
# Convert the clean dataset to csv
df.to_csv("household_clean.csv", index=False) 